# Causal vs. Predictive Churn Model — Simplified Simulation

## What This Notebook Demonstrates

This notebook compares two fundamentally different approaches to targeting customers at risk of churning:

| | Predictive Model | Causal (Uplift) Model |
|---|---|---|
| **Question asked** | *Who will churn?* | *Who will churn **less** because of my action?* |
| **Model type** | Logistic Regression | T-Learner (two Logistic Regressions) |
| **Feedback loop** | Gets confused over time | Remains stable |

---

## The Core Concept: Why Predictive Models Fail in Deployment

When you act on a predictive model (e.g. send discounts to predicted churners), you **change the world**:
- Treated customers stop churning → the model now sees them as "safe"
- The model stops targeting them → they churn again
- A slow performance degradation loop begins

A **causal/uplift model** avoids this by explicitly modelling *treatment effect* — it only targets customers who churn *differently* because of the treatment.

---

## The Causal Structure (DAG)

We intentionally keep this simple — **one independent variable** feeds into churn:

```
satisfaction_score  ──────────────────────────► churn
                                                  ▲
treatment (discount) ─────────────────────────────┘
```

- `satisfaction_score`: A continuous variable (0–10). Lower score → higher churn risk.
- `treatment`: A binary variable (1 = customer received a discount/offer, 0 = did not).
- `churn`: Binary outcome (1 = churned, 0 = stayed).

**No confounders. No mediators. No colliders.** This is by design — the goal is to isolate and clearly demonstrate the predictive vs. causal difference without distractions.

---

## Customer Types (The Uplift Framework)

Every customer falls into one of four categories based on their *true* causal response to treatment:

| Type | Without treatment | With treatment | Should we target? |
|---|---|---|---|
| **Persuadable** | Churns | Stays | ✅ Yes — this is our target |
| **Sure Thing** | Stays | Stays | ❌ No — wastes budget |
| **Lost Cause** | Churns | Churns | ❌ No — no effect |
| **Sleeping Dog** | Stays | Churns | ⚠️ Avoid — treatment backfires |

The predictive model cannot distinguish these. The T-Learner can.

---
## Section 1: Imports & Configuration

We use only standard libraries — no XGBoost needed. Logistic Regression from scikit-learn is intentional: the point is the *framing* of the problem, not model complexity.

**Key config parameters to experiment with:**
- `TREATMENT_EFFECT`: How much a discount reduces churn probability (log-odds). Increase this to make the treatment more powerful.
- `SIM_CYCLES`: Number of deployment/retrain cycles. More cycles = more visible divergence.
- `N_CUSTOMERS`: Customers per cycle. Keep large enough for stable statistics.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION — adjust these to explore different scenarios
# ============================================================
CONFIG = {
    "SEED":             42,

    # Data simulation
    "N_CUSTOMERS":      5000,    # Customers generated per cycle
    "N_HISTORY":        5000,    # Customers in the initial training set

    # Causal effect sizes
    "SATISFACTION_EFFECT": -0.6, # Log-odds effect of satisfaction on churn (negative = higher sat → less churn)
    "TREATMENT_EFFECT":    -1.8, # Log-odds effect of treatment on churn (negative = treatment reduces churn)
    "BASE_CHURN_LOGIT":    0.5,  # Baseline log-odds of churn (intercept)

    # Treatment assignment in historical/random phase
    "HISTORICAL_TREAT_RATE": 0.3,  # 30% randomly treated in history

    # Deployment thresholds
    "PREDICTIVE_THRESHOLD":  0.45,  # Treat if predicted churn prob >= this
    "UPLIFT_THRESHOLD":     -0.05,  # Treat if estimated uplift <= this (negative = treatment helps)

    # Simulation
    "SIM_CYCLES":       10,     # Number of deploy → observe → retrain cycles
    "EPSILON":          0.05,   # 5% random exploration (prevents total data collapse)
}

np.random.seed(CONFIG["SEED"])
print("Configuration loaded.")
print(f"  Simulation: {CONFIG['SIM_CYCLES']} deployment cycles, {CONFIG['N_CUSTOMERS']:,} customers each")
print(f"  Treatment effect (log-odds): {CONFIG['TREATMENT_EFFECT']} → approx. {1 - 1/(1+np.exp(-CONFIG['TREATMENT_EFFECT'])):.0%} churn reduction for average customer")

---
## Section 2: Data Generating Process (DGP)

This is the **ground truth** of our simulated world. In a real scenario you would never have access to this — but here we define it explicitly so we can measure how well each model recovers the true causal structure.

### How churn is generated

The true churn probability is:

$$P(\text{churn}) = \sigma(\text{intercept} + \beta_{\text{sat}} \cdot \text{satisfaction} + \beta_{\text{treat}} \cdot \text{treatment})$$

Where $\sigma$ is the sigmoid function. This is the **exact** data-generating process — no hidden variables, no confounders.

In [ ]:
# ============================================================
# DATA GENERATING PROCESS
# ============================================================

def sigmoid(x):
    """Converts log-odds to probability."""
    return 1 / (1 + np.exp(-x))

def generate_customers(n, treat_rate=None, treatment_array=None):
    """
    Generates n customers with:
      - satisfaction_score: drawn from Normal(5, 2), clipped to [0, 10]
      - treatment: binary, either passed in or assigned randomly at treat_rate
      - churn: binary outcome derived from the true causal model
      - tau_true: true individual treatment effect (CATE) — hidden from models
    """
    # Independent variable: satisfaction score (0–10 scale)
    satisfaction = np.clip(np.random.normal(5, 2, n), 0, 10)

    # Treatment assignment
    if treatment_array is not None:
        treatment = treatment_array.astype(int)
    elif treat_rate is not None:
        treatment = np.random.binomial(1, treat_rate, n)
    else:
        treatment = np.zeros(n, dtype=int)

    # True churn probability (the causal ground truth)
    logit_churn = (
        CONFIG["BASE_CHURN_LOGIT"]
        + CONFIG["SATISFACTION_EFFECT"] * satisfaction
        + CONFIG["TREATMENT_EFFECT"] * treatment
    )
    p_churn = sigmoid(logit_churn)
    churn = np.random.binomial(1, p_churn)

    # True individual treatment effect (tau): how much would churn prob change if treated?
    # This is the counterfactual quantity — unobservable in practice.
    p_churn_if_treated    = sigmoid(CONFIG["BASE_CHURN_LOGIT"] + CONFIG["SATISFACTION_EFFECT"] * satisfaction + CONFIG["TREATMENT_EFFECT"])
    p_churn_if_untreated  = sigmoid(CONFIG["BASE_CHURN_LOGIT"] + CONFIG["SATISFACTION_EFFECT"] * satisfaction)
    tau_true = p_churn_if_treated - p_churn_if_untreated  # negative = treatment reduces churn

    return pd.DataFrame({
        "satisfaction":  satisfaction,
        "treatment":     treatment,
        "churn":         churn,
        "p_churn":       p_churn,
        "tau_true":      tau_true,   # ground truth CATE (hidden from models)
    })


# Quick sanity check
test_df = generate_customers(10000, treat_rate=CONFIG["HISTORICAL_TREAT_RATE"])
print("Sanity check on DGP:")
print(f"  Overall churn rate:       {test_df.churn.mean():.1%}")
print(f"  Churn rate (untreated):   {test_df[test_df.treatment==0].churn.mean():.1%}")
print(f"  Churn rate (treated):     {test_df[test_df.treatment==1].churn.mean():.1%}")
print(f"  Avg true treatment effect (tau): {test_df.tau_true.mean():.3f}  (negative = treatment reduces churn)")

---
## Section 3: Model Definitions

### Predictive Model (Logistic Regression)
Trained on `[satisfaction, treatment]` → predicts `churn`. 
At deployment, it uses `treatment=0` for scoring (since no treatment has been applied yet — it's predicting who *would* churn without intervention). It then treats anyone above the threshold.

**The flaw:** When it treats high-risk customers and they don't churn, the next training batch contains many treated, non-churning customers. The model learns that these customers are *safe* — and stops targeting them.

### Causal T-Learner (Two Logistic Regressions)
Trains **two separate models**:
- `m1`: trained only on **treated** customers → learns P(churn | treated)
- `m0`: trained only on **untreated** customers → learns P(churn | untreated)

At deployment, the estimated uplift for each customer is:
$$\hat{\tau} = \hat{m}_1(\text{satisfaction}) - \hat{m}_0(\text{satisfaction})$$

We treat customers where $\hat{\tau}$ is sufficiently negative (i.e. treatment is predicted to help them meaningfully).

In [ ]:
# ============================================================
# MODEL DEFINITIONS
# ============================================================

FEATURES = ["satisfaction"]  # Only one independent variable — intentionally simple


def train_predictive_model(df):
    """
    Standard logistic regression: predicts P(churn) from satisfaction + treatment.
    Returns the fitted model and its holdout F1 score.
    """
    # The predictive model sees both satisfaction AND treatment
    X = df[["satisfaction", "treatment"]]
    y = df["churn"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=CONFIG["SEED"]
    )
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    f1 = f1_score(y_test, model.predict(X_test), zero_division=0)
    return {"model": model, "f1_test": f1}


def train_tlearner(df):
    """
    T-Learner: two separate logistic regressions — one per treatment arm.
    m1 learns from treated customers; m0 from untreated customers.
    Returns both models and combined holdout F1.
    """
    df1 = df[df.treatment == 1].copy()
    df0 = df[df.treatment == 0].copy()

    X1_train, X1_test, y1_train, y1_test = train_test_split(
        df1[FEATURES], df1["churn"], test_size=0.2, random_state=CONFIG["SEED"]
    )
    X0_train, X0_test, y0_train, y0_test = train_test_split(
        df0[FEATURES], df0["churn"], test_size=0.2, random_state=CONFIG["SEED"]
    )

    m1 = LogisticRegression(max_iter=1000)
    m0 = LogisticRegression(max_iter=1000)
    m1.fit(X1_train, y1_train)
    m0.fit(X0_train, y0_train)

    # Combined test F1: each arm scored by its own model
    test_combined = pd.concat([
        pd.DataFrame({"X": list(X1_test.values), "y": y1_test.values, "t": 1}),
        pd.DataFrame({"X": list(X0_test.values), "y": y0_test.values, "t": 0})
    ])
    preds = [
        m1.predict([row["X"]])[0] if row["t"] == 1 else m0.predict([row["X"]])[0]
        for _, row in test_combined.iterrows()
    ]
    f1 = f1_score(test_combined["y"], preds, zero_division=0)

    return {"m1": m1, "m0": m0, "f1_test": f1}


def predict_uplift(m1, m0, X):
    """
    Estimates the Conditional Average Treatment Effect (CATE) per customer.
    tau_hat = P(churn | treated) - P(churn | untreated)
    Negative values = treatment reduces churn = customer worth targeting.
    """
    p1 = m1.predict_proba(X)[:, 1]
    p0 = m0.predict_proba(X)[:, 1]
    return p1 - p0


print("Model definitions ready.")
print("  - Predictive: LogisticRegression on [satisfaction, treatment] → predict churn")
print("  - T-Learner:  Two LogisticRegressions on [satisfaction] — one per treatment arm")

---
## Section 4: Initial Training Data

We generate a batch of historical customers where treatment was assigned **randomly** (like an A/B test). This random assignment is important — it gives both models an unbiased starting point where treated and untreated groups are comparable.

In [ ]:
# ============================================================
# SECTION 4: INITIAL TRAINING DATA
# ============================================================

print("Generating initial historical training data (random treatment assignment)...")

initial_data = generate_customers(
    CONFIG["N_HISTORY"],
    treat_rate=CONFIG["HISTORICAL_TREAT_RATE"]
)

print(f"  Customers: {len(initial_data):,}")
print(f"  Churn rate: {initial_data.churn.mean():.1%}")
print(f"  Treatment rate: {initial_data.treatment.mean():.1%}")
print(f"  Treated churn rate:   {initial_data[initial_data.treatment==1].churn.mean():.1%}")
print(f"  Untreated churn rate: {initial_data[initial_data.treatment==0].churn.mean():.1%}")

# Train initial models on the historical data
predictive_models = train_predictive_model(initial_data)
causal_models     = train_tlearner(initial_data)

print(f"\nInitial model performance (holdout F1):")
print(f"  Predictive model F1: {predictive_models['f1_test']:.3f}")
print(f"  T-Learner F1:        {causal_models['f1_test']:.3f}")
print("\n(Note: T-Learner F1 is NOT the right metric for a causal model — see Section 7)")

---
## Section 5: Deployment Loop

This is the core of the simulation. In each cycle:

1. **Score** new customers with each model to decide who to treat
2. **Apply treatment** to selected customers
3. **Observe churn** based on the true DGP (with treatment applied)
4. **Retrain** each model on the accumulated data
5. **Record metrics** — both standard ML metrics and business metrics

The key thing to watch: the **predictive model's training data gets increasingly biased** because it treated high-risk customers → those customers didn't churn → now they look like low-risk customers in the next training round.

In [ ]:
# ============================================================
# SECTION 5: DEPLOYMENT LOOP
# ============================================================

# Storage for metrics across cycles
metrics = []

# Each policy maintains its own rolling training dataset
pred_train_data   = initial_data.copy()
causal_train_data = initial_data.copy()

pred_models   = predictive_models
uplift_models = causal_models

print(f"Starting {CONFIG['SIM_CYCLES']}-cycle deployment simulation...\n")

for cycle in range(1, CONFIG["SIM_CYCLES"] + 1):

    # ── Generate new customers (no treatment applied yet) ──────────────────
    new_customers = generate_customers(CONFIG["N_CUSTOMERS"], treat_rate=0)
    X_score = new_customers[FEATURES]

    # ── PREDICTIVE MODEL: decide whom to treat ─────────────────────────────
    # Score with treatment=0 (pre-treatment churn probability)
    X_score_pred = pd.DataFrame({"satisfaction": X_score["satisfaction"], "treatment": 0})
    pred_churn_prob = pred_models["model"].predict_proba(X_score_pred)[:, 1]
    pred_treat = (pred_churn_prob >= CONFIG["PREDICTIVE_THRESHOLD"]).astype(int)

    # Epsilon-greedy: randomly treat a small fraction to avoid data starvation
    explore_mask = np.random.rand(len(new_customers)) < CONFIG["EPSILON"]
    pred_treat = np.maximum(pred_treat, explore_mask.astype(int))

    # ── CAUSAL MODEL: decide whom to treat ────────────────────────────────
    estimated_uplift = predict_uplift(uplift_models["m1"], uplift_models["m0"], X_score)
    causal_treat = (estimated_uplift <= CONFIG["UPLIFT_THRESHOLD"]).astype(int)

    explore_mask_c = np.random.rand(len(new_customers)) < CONFIG["EPSILON"]
    causal_treat = np.maximum(causal_treat, explore_mask_c.astype(int))

    # ── Observe outcomes under each policy's treatment decisions ───────────
    pred_observed   = generate_customers(CONFIG["N_CUSTOMERS"], treatment_array=pred_treat)
    causal_observed = generate_customers(CONFIG["N_CUSTOMERS"], treatment_array=causal_treat)

    # Share the same underlying customers (same satisfaction scores)
    pred_observed["satisfaction"]   = new_customers["satisfaction"].values
    causal_observed["satisfaction"] = new_customers["satisfaction"].values

    # Recompute churn with the actual satisfaction scores
    def apply_dgp(df):
        logit = (
            CONFIG["BASE_CHURN_LOGIT"]
            + CONFIG["SATISFACTION_EFFECT"] * df["satisfaction"]
            + CONFIG["TREATMENT_EFFECT"] * df["treatment"]
        )
        df["churn"] = np.random.binomial(1, sigmoid(logit.values))
        return df

    pred_observed["treatment"]   = pred_treat
    causal_observed["treatment"] = causal_treat
    pred_observed["satisfaction"]   = new_customers["satisfaction"].values
    causal_observed["satisfaction"] = new_customers["satisfaction"].values
    pred_observed   = apply_dgp(pred_observed)
    causal_observed = apply_dgp(causal_observed)

    # ── Append to each policy's training history ───────────────────────────
    pred_train_data   = pd.concat([pred_train_data,   pred_observed],   ignore_index=True)
    causal_train_data = pd.concat([causal_train_data, causal_observed], ignore_index=True)

    # ── Retrain models ─────────────────────────────────────────────────────
    pred_models   = train_predictive_model(pred_train_data)
    uplift_models = train_tlearner(causal_train_data)

    # ── Business metric: how well does each policy target TRUE persuadables?
    #    True persuadables = customers where tau_true is sufficiently negative
    true_persuadables = (new_customers["tau_true"] <= CONFIG["UPLIFT_THRESHOLD"]).values

    def policy_f1(treat_decisions, true_persuadables):
        """F1 of targeting decisions vs. ground truth persuadables."""
        return f1_score(true_persuadables, treat_decisions, zero_division=0)

    metrics.append({
        "cycle":              cycle,
        # Treatment rates
        "pred_treat_rate":    pred_treat.mean(),
        "causal_treat_rate":  causal_treat.mean(),
        # Churn rates under each policy
        "pred_churn_rate":    pred_observed["churn"].mean(),
        "causal_churn_rate":  causal_observed["churn"].mean(),
        # Holdout ML metric (misleading for causal!)
        "pred_f1_test":       pred_models["f1_test"],
        "causal_f1_test":     uplift_models["f1_test"],
        # Business metric: are we targeting the right people?
        "pred_policy_f1":     policy_f1(pred_treat, true_persuadables),
        "causal_policy_f1":   policy_f1(causal_treat, true_persuadables),
        # Training data composition
        "pred_train_treat_rate":   pred_train_data["treatment"].mean(),
        "causal_train_treat_rate": causal_train_data["treatment"].mean(),
        "pred_train_churn_rate":   pred_train_data["churn"].mean(),
        "causal_train_churn_rate": causal_train_data["churn"].mean(),
    })

    print(f"  Cycle {cycle:2d}/{CONFIG['SIM_CYCLES']}  "
          f"| Pred churn: {pred_observed['churn'].mean():.1%}  "
          f"| Causal churn: {causal_observed['churn'].mean():.1%}  "
          f"| Pred policy F1: {metrics[-1]['pred_policy_f1']:.3f}  "
          f"| Causal policy F1: {metrics[-1]['causal_policy_f1']:.3f}")

df_metrics = pd.DataFrame(metrics)
print("\nSimulation complete.")

---
## Section 6: Results Visualization

Four plots tell the full story:

1. **Policy F1 (Business ROI)** — Are we actually targeting the right customers (true persuadables)? This is the metric that matters for business.
2. **Churn Rate** — What % of customers churn under each policy?
3. **Holdout Test F1** — Standard ML metric. Notice the predictive model's test F1 can look fine even as its business performance degrades — this is the *accuracy trap*.
4. **Training Data Bias** — The treatment rate in each model's training data over time. The predictive model's data becomes increasingly imbalanced.

In [ ]:
# ============================================================
# SECTION 6: RESULTS VISUALIZATION
# ============================================================

fig, axs = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Causal vs. Predictive Model: Deployment Simulation", fontsize=15, fontweight="bold")

cycles = df_metrics["cycle"]
RED,  BLUE = "#e74c3c", "#2980b9"

# ── Plot 1: Business ROI (Policy F1 vs ground truth persuadables) ──────────
ax = axs[0, 0]
ax.plot(cycles, df_metrics["pred_policy_f1"],   color=RED,  marker="o", label="Predictive")
ax.plot(cycles, df_metrics["causal_policy_f1"], color=BLUE, marker="s", label="T-Learner (Causal)")
ax.set_title("① Policy F1: Targeting the Right Customers\n(vs. ground truth persuadables)", fontsize=11)
ax.set_ylabel("F1 Score (higher = better targeting)")
ax.set_xlabel("Deployment Cycle")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

# ── Plot 2: Observed churn rate ─────────────────────────────────────────────
ax = axs[0, 1]
ax.plot(cycles, df_metrics["pred_churn_rate"],   color=RED,  marker="o", label="Predictive")
ax.plot(cycles, df_metrics["causal_churn_rate"], color=BLUE, marker="s", label="T-Learner (Causal)")
ax.set_title("② Observed Churn Rate\n(lower = better business outcome)", fontsize=11)
ax.set_ylabel("Churn Rate")
ax.set_xlabel("Deployment Cycle")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend()
ax.grid(True, alpha=0.3)

# ── Plot 3: Holdout Test F1 (the accuracy trap) ─────────────────────────────
ax = axs[1, 0]
ax.plot(cycles, df_metrics["pred_f1_test"],   color=RED,  marker="o", label="Predictive")
ax.plot(cycles, df_metrics["causal_f1_test"], color=BLUE, marker="s", label="T-Learner (Causal)")
ax.set_title("③ Holdout Test F1: The 'Accuracy Trap'\n(can look good while business suffers)", fontsize=11)
ax.set_ylabel("Test F1 Score")
ax.set_xlabel("Deployment Cycle")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

# ── Plot 4: Training data bias — treatment rate over time ───────────────────
ax = axs[1, 1]
ax.plot(cycles, df_metrics["pred_train_treat_rate"],   color=RED,  marker="o", label="Predictive")
ax.plot(cycles, df_metrics["causal_train_treat_rate"], color=BLUE, marker="s", label="T-Learner (Causal)")
ax.axhline(CONFIG["HISTORICAL_TREAT_RATE"], color="gray", linestyle="--", alpha=0.6,
           label=f"Initial random rate ({CONFIG['HISTORICAL_TREAT_RATE']:.0%})")
ax.set_title("④ Training Data: Treatment Rate Over Time\n(predictive model's data gets biased)", fontsize=11)
ax.set_ylabel("Fraction of Treated Customers in Training Data")
ax.set_xlabel("Deployment Cycle")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/simulation_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")

---
## Section 7: Final Summary & Key Insights

### Why the predictive model degrades

The predictive model creates a **feedback loop**:

```
Model targets high-risk customers
       ↓
They receive treatment → don't churn
       ↓
Next training batch: these customers are labeled "safe"
       ↓
Model learns they're low-risk → stops targeting them
       ↓
They churn again → cycle repeats
```

### Why the T-Learner stays robust

The T-Learner doesn't ask *"will they churn?"* — it asks *"would treatment change whether they churn?"* This is stable even when the treated customers' outcomes change, because the model explicitly conditions on treatment status.

### The Accuracy Trap

Plot ③ demonstrates a critical insight: the predictive model's **test F1 can remain high** even while its targeting is getting worse. This is because test F1 measures prediction accuracy on observed data — data that has already been biased by the model's own actions. Standard ML evaluation metrics cannot detect this kind of deployment failure.

In [ ]:
# ============================================================
# SECTION 7: FINAL SUMMARY
# ============================================================

first = df_metrics.iloc[0]
last  = df_metrics.iloc[-1]

print("=" * 60)
print("FINAL BUSINESS IMPACT SUMMARY")
print("=" * 60)

print(f"\n{'Metric':<40} {'Cycle 1':>10} {'Final':>10}")
print("-" * 62)

metrics_to_show = [
    ("Predictive: Policy F1 (targeting accuracy)",  "pred_policy_f1"),
    ("Causal:     Policy F1 (targeting accuracy)",  "causal_policy_f1"),
    ("Predictive: Churn rate",                       "pred_churn_rate"),
    ("Causal:     Churn rate",                       "causal_churn_rate"),
    ("Predictive: Holdout Test F1",                  "pred_f1_test"),
    ("Causal:     Holdout Test F1",                  "causal_f1_test"),
]

for label, col in metrics_to_show:
    v1 = first[col]
    vf = last[col]
    arrow = "↑" if vf > v1 else "↓" if vf < v1 else "→"
    # For churn rate, down is good; for F1, up is good
    print(f"{label:<40} {v1:>10.3f} {vf:>10.3f} {arrow}")

print("\n" + "=" * 60)
print("KEY TAKEAWAY")
print("=" * 60)

pred_policy_change  = last["pred_policy_f1"]  - first["pred_policy_f1"]
causal_policy_change = last["causal_policy_f1"] - first["causal_policy_f1"]

print(f"""
Predictive model targeting accuracy changed by: {pred_policy_change:+.3f}
Causal model targeting accuracy changed by:     {causal_policy_change:+.3f}

The predictive model's holdout Test F1 {'remained stable' if abs(last['pred_f1_test'] - first['pred_f1_test']) < 0.05 else 'changed'}
even as its business targeting {'degraded' if pred_policy_change < 0 else 'changed'}.
This is the 'Accuracy Trap': standard ML metrics do not measure
whether a model is making correct *causal* decisions.
""")

---
## 🔬 Extensions: Things to Try

Now that you have a working base, here are natural next steps to explore:

### Easy (change CONFIG values)
1. **Increase `TREATMENT_EFFECT`** (e.g. to -3.0): The treatment becomes very powerful — does the causal model pull further ahead?
2. **Increase `SIM_CYCLES`** to 20 or 30: The divergence between models typically becomes clearer over more cycles.
3. **Set `EPSILON = 0.0`**: Remove exploration entirely — does the predictive model collapse faster?

### Medium (add a second independent variable)
4. **Add `tenure` as a second feature**: Longer-tenured customers may respond differently to discounts. This is a natural extension of the DAG without adding confounders.

### Advanced (structural changes)
5. **Add a simple confounder**: E.g. `high_value` customers are both more likely to receive treatment *and* less likely to churn. This introduces selection bias — a perfect teaching moment.
6. **Replace Logistic Regression with a tree-based model** (e.g. `GradientBoostingClassifier`) to see whether model complexity changes the pattern.